In [12]:
!pip install alpha_vantage

In [13]:
import pandas as pd
import requests
import json
from alpha_vantage.timeseries import TimeSeries
from dotenv import load_dotenv
import os

Carregando a Chave do arquivo .env

In [14]:
load_dotenv()
api_key = os.getenv('ALPHA_VANTAGE_API_KEY')
print("API Key loaded successfully" if api_key else "API Key not found")

url_base = "https://www.alphavantage.co/query"

API Key loaded successfully


Colocar numa função para chamar em diferentes tempos

In [15]:
# Para pegar a API key, acesse https://www.alphavantage.co/support/#api-key
symbol = 'AAPL'
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=apple&interval=5min&apikey=api_key'
r = requests.get(url)

print(r)

<Response [200]>


In [ ]:
def get_data(symbol, interval='5min', api_key=api_key):

    params = {
        'function': 'TIME_SERIES_INTRADAY',
        'symbol': symbol,
        'interval': interval,
        'outputsize': 'full',
        'apikey': api_key
    }
    response = requests.get(url_base, params=params)
    json_data = response.json()
    ts_key = f'Time Series ({interval})'
    data = json_data.get(ts_key, {})
    df = pd.DataFrame.from_dict(data, orient='index')
    df.index = pd.to_datetime(df.index)
    return df.astype(float).sort_index()

df = get_data(symbol=symbol, interval="5min", api_key=api_key)
df.head()

,1. open,2. high,3. low,4. close,5. volume
2025-03-31 04:00:00,216.90,216.90,215.35,215.86,12498.0
2025-03-31 04:05:00,215.85,216.89,215.61,216.47,10160.0
2025-03-31 04:10:00,216.36,216.50,216.02,216.45,2282.0
2025-03-31 04:15:00,216.38,216.85,216.11,216.17,7199.0
2025-03-31 04:20:00,216.12,216.18,215.62,215.77,4638.0


In [23]:
df.to_parquet(f'data/{symbol}.parquet')